### Klasse

Eine Klasse `Foo` stellt Methoden, also Funktionen mit Signaturen der Form `f(self, ...)`
zur Verfügung. Ist `foo` eine Instanz der Klasse `Foo`,
so hat `foo.f(...)` den gleichen Effekt wie `Foo.f(foo, ...)`.
Das erlaubt der Methode `f` auf Attribute der Instanz `foo` zuzugreifen.

In Python ist dieser Mechanismus wie folgt implementiert.
Wird versucht, mit `foo.bar` auf das Attribut `bar` zuzugreifen, dann geschieht folgendes:

- Ist `bar` ein Attribut der Instanz `foo` wird direkt `foo.bar` geliefert.
- Ist `bar` **kein** Attribut der Instanz `foo` wird geschaut, ob
  die Klasse `Foo` ein Attribut `bar` hat.
  - Falls nicht, wird ein `AttributError` ausgelöst.
  - Fall ja, wird geschaut, ob das Attribut `bar` **callable** ist (z.B. eine Funktion).
     - Falls nicht, wird `Foo.bar` geliefert.
     - Falls ja, wird die Funktion
       `lambda *args, self=foo, **kwargs: bar(self, *args, **kwargs)` geliefert.  
       Das ist die Funktion `Foo.bar`, wo nun dem Argument `self` dauerhaft der Wert `foo` zugeordnet wurde.
       Man sagt, auch `self` ist nun an das Objekt `foo` gebunden.

**Zur Erinnerung:**
`f = lambda *args, self=foo, **kwargs: f(self, *args, **kwargs)` ist im Wesentlichen eine Kurzform von  
```python
def f(*args, self=foo, **kwargs):
    return f(self, *args, **kwargs)
```


**Beachte:** Die Funktion `f` nimmt kein oder mehrere positionale (normale) Argumente,
dann folgt das (optionale, aber stets wegzulassende) Argument `self`, welchem der Default-Wert `foo` (eine Klasseninstanz) zugewiesen ist,
dann kein oder mehrere Keyword-Argumente (Argumente der Form `<variabelname>=<Wert>`).

In [103]:
class Foo:
    # pass
    x = 42

    def print_x(self):
        print(self.x)

    def __repr__(self):
        return 'Foo()'

In [104]:
foo = Foo()
foo.x

42

In [105]:
foo.print_x()

42


In [110]:
f = lambda *args, self=foo, **kwargs: Foo.print_x(self, *args, **kwargs)
f()

42


In [111]:
Foo.bar

AttributeError: type object 'Foo' has no attribute 'bar'

### Beispiel einer Klasse

In [114]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def birthday(self):
        self.age += 1

    def __repr__(self):
        return f'Person({self.name}, {self.age})'

In [115]:
bob = Person('Bob', 20)
bob

Person(Bob, 20)

In [116]:
bob.birthday()
bob

Person(Bob, 21)

### Handgestrickte Klassen

Nachstehend implementiere wir einen solchen Mechanismus selber.  
Dies soll vorallem der Illustratiin dienen.



Wir wollen eine Klasse als Dictionary implementieren.
Dieser Dictionary speichert die Daten und Funktionen der Klasse.
Die speziellen Schlüssel `__name__` und `__init__` speichern den Klasssennamen und die
Initialisierungsfunktion.

Im Unterschied zu richtigen Klassen können wir Attribute nicht mit die dot-Notation ansprechen, sondern
müssen die übliche Dict-Syntax nutzen, um den Wert eines Schlüssels zu erhalten.  
Deshalb steht in der nächsten Zelle z.B. `self['name'] = name` anstelle von `self.name = name`.

In [117]:
def __init__(self, name, age):
    self['name'] = name
    self['age'] = age


def birthday(self):
    self['age'] += 1


def show(self):
    print(f'Person({self['name']}, {self['age']})')


Person = {
    '__name__': 'Person',
    '__init__': __init__,
    'birthday': birthday,
    'show': show,
}

Eine Instanz `bob` dieser Klasse wird dann wie folgt erstellt:
- Wir erstellen einen fast leeren Dict `bob = {'__class__': cls}`, der zunächst nur den Dict mit den Klassenatttributen
(die Klasse selbst) speichert.
- Dann wird die Initialisierungsfunktion  `Person['__init__'](bob)` ausgeführt, welche
  den Dict `bob` dann initialisiert (weitere Schlüssel-Wert Paare hinzufügt.

In [118]:
bob = {'__class__': Person}
Person['__init__'](bob, name='Bob', age=20)

Person['birthday'](bob)
Person['show'](bob)

Person(Bob, 21)


Die Methode `birthday` dieser Klasse Person erhalten wir dann, indem wir
das Argument `self` der Funktion `Person['birthday'](self)` an `bob` binden.

In [119]:
birthday = lambda *args, self=bob, **kwargs: Person['birthday'](self, *args, **kwargs)
show = lambda *args, self=bob, **kwargs: Person['show'](self, *args, **kwargs)

birthday()
show()

Person(Bob, 22)


Die Funktionen `make_instance` und `get_attr` erledigen die Instatierung, bew. den Attributzugriff.

In [120]:
def make_instance(cls, *args, **kwargs):
    d = {'__class__': cls}

    __init__ = cls.get('__init__')
    if __init__:
        __init__(d, *args, **kwargs)

    return d


def get_attr(instance, attr):
    'Zugriff auf Attribute einer Instanz'
    assert '__class__' in instance, 'did not get an instance of a class'

    if attr in instance:
        return instance[attr]

    elif attr in instance['__class__']:
        obj = instance['__class__'][attr]
        if not callable(obj):
            return obj
        else:
            return lambda *args, **kwargs: obj(instance, *args, **kwargs)

    else:
        cls = instance['__class__']
        raise AttributeError(f"type object '{cls['__name__']}' has no attribute '{attr}'")

In [123]:
anna = make_instance(Person, name='Anna', age=30)  # anna = Anna(name='Anna', age=30)
get_attr(anna, 'show')()  # anna.show()

Person(Anna, 30)


In [124]:
get_attr(anna, 'birthday')()  # anna.birthday()
get_attr(anna, 'show')()  # anna.show() 

Person(Anna, 31)
